# Simulated Price Comparison Engine (Using eBay Browse API Sandbox)

## 1. Introduction

#### This notebook will:

##### 1. Authenticate with sandbox credentials
##### 2. Perform searches via the Browse API sandbox endpoint
##### 3. Parse sample listings (sandbox-generated)
##### 4. Analyze average price, brand trends, and listing conditions
##### 5. Visualize results—all with sandbox test data only

## 2. Load Credentials (from .env)

In [ ]:
from dotenv import load_dotenv
import requests
import base64
import os

# Load environment variables from .env file
load_dotenv()

EBAY_CLIENT_ID = os.getenv("EBAY_CLIENT_ID")
EBAY_CLIENT_SECRET = os.getenv("EBAY_CLIENT_SECRET")

## 3. Authenticate (Sandbox Token)

In [ ]:
def get_sandbox_access_token():
    url = "https://api.sandbox.ebay.com/identity/v1/oauth2/token"
    auth = base64.b64encode(f"{EBAY_CLIENT_ID}:{EBAY_CLIENT_SECRET}".encode()).decode()
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {auth}"
    }
    data = {"grant_type": "client_credentials", "scope": "https://api.ebay.com/oauth/api_scope"}
    resp = requests.post(url, headers=headers, data=data)
    print("Status:", resp.status_code, resp.json())
    return resp.json().get("access_token")

access_token = get_sandbox_access_token()

## 4. Search via Browse API (Sandbox)

In [ ]:
def sandbox_search_items(keyword, limit=20):
    url = "https://api.sandbox.ebay.com/buy/browse/v1/item_summary/search"
    headers = {"Authorization": f"Bearer {access_token}"}
    params = {"q": keyword, "limit": limit}
    resp = requests.get(url, headers=headers, params=params)
    print("Search status:", resp.status_code)
    return resp.json().get("itemSummaries", [])

items = sandbox_search_items("phone", limit=20)


## 5. Parse Listing Data

In [ ]:
import pandas as pd

def parse_sandbox_items(items):
    rows = []
    for item in items:
        rows.append({
            "itemId": item.get("itemId"),
            "title": item.get("title"),
            "price": float(item["price"]["value"]),
            "currency": item["price"]["currency"],
            "condition": item.get("condition"),
            "brand": next((asp["label"] for asp in item.get("aspectDistribution", []) if asp["field"] == "Brand"), None)
        })
    return pd.DataFrame(rows)

df = parse_sandbox_items(items)
df.head()


## 6. Analysis & Visualizations

In [ ]:
print("Average Price:", df["price"].mean())

import seaborn as sns
import matplotlib.pyplot as plt

# Brand distribution
sns.countplot(data=df, x="brand")
plt.title("Brand Frequency in Results");

# Price distribution
sns.histplot(df["price"], bins=10)
plt.title("Price Distribution")


## 7. Documentation